In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install transformers datasets evaluate accelerate scikit-learn --quiet

import torch
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
print(f'GPU      : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00
PyTorch  : 2.11.0+cu128
CUDA     : True
GPU      : Tesla T4


In [ ]:
# 15 final categories
CATEGORIES = [
    "Politics",
    "Business & Economy",
    "Sports",
    "Entertainment",
    "Science & Technology",
    "Health & Medicine",
    "Crime & Law",
    "Education",
    "Environment",
    "Lifestyle",
    "Travel",
    "Religion & Culture",
    "Parenting",
    "Opinion & Editorial",
    "International"
]

# 40 categories →  15
RAW_TO_CATEGORY = {
    # Politics
    "POLITICS":         "Politics",
    "U.S. NEWS":        "Politics",

    # International
    "WORLD NEWS":       "International",
    "WORLDPOST":        "International",
    "THE WORLDPOST":    "International",

    # Business
    "BUSINESS":         "Business & Economy",
    "MONEY":            "Business & Economy",

    # Sports
    "SPORTS":           "Sports",

    # Entertainment
    "ENTERTAINMENT":    "Entertainment",
    "ARTS":             "Entertainment",
    "ARTS & CULTURE":   "Entertainment",
    "CULTURE & ARTS":   "Entertainment",
    "COMEDY":           "Entertainment",
    "MEDIA":            "Entertainment",
    "WEIRD NEWS":       "Entertainment",

    # Science & Technology
    "SCIENCE":          "Science & Technology",
    "TECH":             "Science & Technology",

    # Health
    "WELLNESS":         "Health & Medicine",
    "HEALTHY LIVING":   "Health & Medicine",

    # Crime
    "CRIME":            "Crime & Law",

    # Education
    "EDUCATION":        "Education",
    "COLLEGE":          "Education",

    # Environment
    "ENVIRONMENT":      "Environment",
    "GREEN":            "Environment",

    # Lifestyle
    "STYLE":            "Lifestyle",
    "STYLE & BEAUTY":   "Lifestyle",
    "HOME & LIVING":    "Lifestyle",
    "FOOD & DRINK":     "Lifestyle",
    "TASTE":            "Lifestyle",
    "FIFTY":            "Lifestyle",
    "GOOD NEWS":        "Lifestyle",
    "WOMEN":            "Lifestyle",

    # Travel
    "TRAVEL":           "Travel",

    # Religion
    "RELIGION":         "Religion & Culture",

    # Parenting
    "PARENTING":        "Parenting",
    "PARENTS":          "Parenting",

    # Opinion
    "IMPACT":           "Opinion & Editorial",
    "BLACK VOICES":     "Opinion & Editorial",
    "QUEER VOICES":     "Opinion & Editorial",
    "LATINO VOICES":    "Opinion & Editorial",
    "DIVORCE":          "Opinion & Editorial",
    "WEDDINGS":         "Opinion & Editorial",
}

# Label encode
LABEL2ID = {cat: i for i, cat in enumerate(CATEGORIES)}
ID2LABEL = {i: cat for i, cat in enumerate(CATEGORIES)}

print(f'Total categories: {len(CATEGORIES)}')
print(f'Dataset categories mapped: {len(RAW_TO_CATEGORY)}')
for i, cat in enumerate(CATEGORIES):
    print(f'  {i:2d} → {cat}')

Total categories: 15
Dataset categories mapped: 42
   0 → Politics
   1 → Business & Economy
   2 → Sports
   3 → Entertainment
   4 → Science & Technology
   5 → Health & Medicine
   6 → Crime & Law
   7 → Education
   8 → Environment
   9 → Lifestyle
  10 → Travel
  11 → Religion & Culture
  12 → Parenting
  13 → Opinion & Editorial
  14 → International


In [ ]:
from datasets import load_dataset
import pandas as pd

print('Loading dataset...')
raw = load_dataset('heegyu/news-category-dataset')
df  = pd.DataFrame(raw['train'])

print(f'Raw dataset size: {len(df)}')
print(f'Raw categories  : {df["category"].nunique()}')

# Map to 15 categories
df['label_name'] = df['category'].map(RAW_TO_CATEGORY)

# Drop unmapped rows
df = df.dropna(subset=['label_name'])
print(f'After mapping   : {len(df)} rows')

# Combine headline + description as input text
df['text'] = df['headline'] + '. ' + df['short_description']
df['text'] = df['text'].str.strip()

# Drop empty
df = df[df['text'].str.len() > 20]

# Add integer label
df['label'] = df['label_name'].map(LABEL2ID)

print(f'Final dataset   : {len(df)} rows')
print(f'\nCategory distribution:')
print(df['label_name'].value_counts().to_string())

Loading dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/101 [00:00<?, ?B/s]

data.json:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/209527 [00:00<?, ? examples/s]

Raw dataset size: 209527
Raw categories  : 42
After mapping   : 209527 rows
Final dataset   : 209119 rows

Category distribution:
label_name
Politics                36932
Entertainment           32248
Lifestyle               31152
Health & Medicine       24576
Opinion & Editorial     22604
Parenting               12744
Travel                   9884
International            9525
Business & Economy       7734
Sports                   5070
Science & Technology     4308
Environment              4060
Crime & Law              3559
Religion & Culture       2571
Education                2152


In [ ]:
from sklearn.model_selection import train_test_split
from datasets import Dataset

train_df, val_df = train_test_split(
    df[['text', 'label', 'label_name']],
    test_size=0.1,
    random_state=42,
    stratify=df['label']
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

# Converted to HuggingFace Dataset
train_ds = Dataset.from_pandas(train_df[['text', 'label']])
val_ds   = Dataset.from_pandas(val_df[['text', 'label']])

print(f'Train : {len(train_ds)} samples')
print(f'Val   : {len(val_ds)} samples')
print(f'\nSample:')
print(f'  Text  : {train_ds[0]["text"]}')
print(f'  Label : {train_ds[0]["label"]} → {ID2LABEL[train_ds[0]["label"]]}')

Train : 188207 samples
Val   : 20912 samples

Sample:
  Text  : New York Times Opinion Page To Publish Letters From Trump Supporters. The newspaper announced that the featured letters are meant to balance out many of the anti-Trump columns.
  Label : 3 → Entertainment


In [ ]:


from transformers import DistilBertTokenizerFast
import datasets.formatting.torch_formatter as tf
import torch
import numpy as np

MODEL_NAME = 'distilbert-base-uncased'
tokenizer  = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

# Tokenize
def tokenize(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        padding='max_length',
        max_length=128
    )

train_tok = train_ds.map(tokenize, batched=True, batch_size=512)
val_tok   = val_ds.map(tokenize,   batched=True, batch_size=512)

def patched_tensorize(self, value):
    if isinstance(value, torch.Tensor):
        return value
    return torch.tensor(np.array(value))

tf.TorchFormatter._tensorize = patched_tensorize

# Now safe to set_format
train_tok.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
val_tok.set_format('torch',   columns=['input_ids', 'attention_mask', 'label'])

print(f'✓ Tokenized — train: {len(train_tok)} | val: {len(val_tok)}')
print(f'  Input shape: {train_tok[0]["input_ids"].shape}')

Map:   0%|          | 0/188207 [00:00<?, ? examples/s]

Map:   0%|          | 0/20912 [00:00<?, ? examples/s]

✓ Tokenized — train: 188207 | val: 20912
  Input shape: torch.Size([128])


In [ ]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(CATEGORIES),
    id2label=ID2LABEL,
    label2id=LABEL2ID
)

print(f'✓ Model loaded')
print(f'  Parameters : {sum(p.numel() for p in model.parameters()):,}')
print(f'  Num labels : {len(CATEGORIES)}')

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✓ Model loaded
  Parameters : 66,965,007
  Num labels : 15


In [ ]:
import evaluate
import numpy as np
from transformers import TrainingArguments, Trainer

OUTPUT_DIR  = '/content/drive/MyDrive/Newspaper_Project/news_classifier'

accuracy = evaluate.load('accuracy')

def compute_metrics(eval_pred):

    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return accuracy.compute(predictions=preds, references=labels)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=500,              # ← changed from warmup_ratio

    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',

    logging_steps=100,
    report_to='none',
    fp16=True,
    dataloader_num_workers=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    compute_metrics=compute_metrics,
)

print('✓ Trainer ready')
print(f'  Epochs     : 3')
print(f'  Batch size : 32')
print(f'  LR         : 2e-5')
print(f'  Output     : {OUTPUT_DIR}')

✓ Trainer ready
  Epochs     : 3
  Batch size : 32
  LR         : 2e-5
  Output     : /content/drive/MyDrive/Newspaper_Project/news_classifier


In [ ]:
print('Starting training...')
trainer.train()
print('\n✓ Training complete!')

Starting training...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.758394,0.722064,0.768124
2,0.570278,0.695737,0.779696
3,0.454366,0.702725,0.782565


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✓ Training complete!


In [ ]:
from sklearn.metrics import classification_report
import numpy as np

preds_output = trainer.predict(val_tok)
preds  = np.argmax(preds_output.predictions, axis=1)
labels = preds_output.label_ids

print('========== EVALUATION RESULTS ==========')
print(classification_report(
    labels, preds,
    target_names=CATEGORIES,
    digits=3
))

========== EVALUATION RESULTS ==========
                      precision    recall  f1-score   support

            Politics      0.812     0.850     0.831      3693
  Business & Economy      0.642     0.624     0.633       773
              Sports      0.813     0.799     0.806       507
       Entertainment      0.794     0.785     0.789      3225
Science & Technology      0.648     0.610     0.628       431
   Health & Medicine      0.804     0.833     0.818      2458
         Crime & Law      0.655     0.657     0.656       356
           Education      0.598     0.526     0.559       215
         Environment      0.616     0.648     0.631       406
           Lifestyle      0.824     0.800     0.812      3115
              Travel      0.852     0.861     0.857       988
  Religion & Culture      0.677     0.661     0.669       257
           Parenting      0.773     0.812     0.792      1274
 Opinion & Editorial      0.763     0.710     0.735      2261
       International      0.

In [ ]:
import json, os

SAVE_DIR = '/content/drive/MyDrive/Newspaper_Project/news_classifier/final_model'
os.makedirs(SAVE_DIR, exist_ok=True)

# Save model + tokenizer
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

# Save label maps
with open(f'{SAVE_DIR}/label_map.json', 'w') as f:
    json.dump({
        'id2label': ID2LABEL,
        'label2id': LABEL2ID,
        'categories': CATEGORIES
    }, f, indent=2)

print('✓ Saved to Drive:')
for f in os.listdir(SAVE_DIR):
    size = os.path.getsize(f'{SAVE_DIR}/{f}') / (1024*1024)
    print(f'  {f}  —  {size:.1f} MB')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Saved to Drive:
  config.json  —  0.0 MB
  model.safetensors  —  255.5 MB
  training_args.bin  —  0.0 MB
  tokenizer_config.json  —  0.0 MB
  tokenizer.json  —  0.7 MB
  label_map.json  —  0.0 MB


In [ ]:
from transformers import pipeline

# Load saved model for inference
classifier = pipeline(
    'text-classification',
    model=SAVE_DIR,
    tokenizer=SAVE_DIR,
    device=0 if torch.cuda.is_available() else -1
)

# Test with sample texts similar to what OCR would give
test_texts = [
    # Politics
    "Prime Minister announced new economic policy today in Parliament. The government will increase spending on infrastructure and reduce corporate taxes to boost GDP growth.",

    # Sports
    "India beat Australia by 6 wickets in the third Test match at Melbourne. Virat Kohli scored a brilliant century and Jasprit Bumrah took 5 wickets in the second innings.",

    # Crime
    "Police arrested three suspects in connection with the bank robbery that took place on Tuesday. The accused were caught near the state border with cash worth Rs 50 lakh.",

    # Health
    "Doctors at AIIMS have developed a new treatment for dengue fever that reduces recovery time by 40 percent. The drug will undergo clinical trials in the next six months.",

    # Business
    "Reliance Industries reported a net profit of Rs 15,000 crore in Q2, beating analyst estimates. The company announced expansion plans for its retail and telecom divisions."
]

print('========== INFERENCE TEST ==========')
for text in test_texts:
    result = classifier(text[:512], truncation=True)[0]
    print(f'Category  : {result["label"]}')
    print(f'Confidence: {result["score"]:.2%}')
    print(f'Text      : {text[:80]}...')
    print()

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

========== INFERENCE TEST ==========
Category  : International
Confidence: 90.09%
Text      : Prime Minister announced new economic policy today in Parliament. The government...

Category  : Sports
Confidence: 81.96%
Text      : India beat Australia by 6 wickets in the third Test match at Melbourne. Virat Ko...

Category  : International
Confidence: 70.31%
Text      : Police arrested three suspects in connection with the bank robbery that took pla...

Category  : Health & Medicine
Confidence: 92.96%
Text      : Doctors at AIIMS have developed a new treatment for dengue fever that reduces re...

Category  : Business & Economy
Confidence: 87.72%
Text      : Reliance Industries reported a net profit of Rs 15,000 crore in Q2, beating anal...

